# Analysis of data for the Essential FFPE Panel

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.metrics import cohen_kappa_score
from sklearn.neighbors import NearestNeighbors
from scipy.stats import gaussian_kde
from sklearn.mixture import GaussianMixture

from tqdm import trange

### BatchDetect module imports

Import project-specific helpers from the `batchdetect` package:

- `load_thal_cross_lot_covs` and related loaders for Thalassemia data.
- `HeavyMixture` and `parametric_bootstrap_lrt` for mixture modeling and
  parametric bootstrap-based likelihood ratio tests.
- Correlation-based clustering utilities:
  `cluster_hierarchical_corr`, `cluster_spectral_corr`,
  `cluster_pca_kmeans_corr`.
- Correlation preprocessing helpers: `normalize_mat` and `get_correlations`.

These functions implement the main batch-detection and clustering methods used
throughout the analysis.


In [2]:
from batchdetect.loader import load_thal_cross_lot_covs
from batchdetect.mixture import HeavyMixture,parametric_bootstrap_lrt

df_thal_likelihoods = pd.read_csv('../likelihoods_thal131.csv')
print(df_thal_likelihoods.head())
snames = df_thal_likelihoods['Sample Name'].values
likelihoods = df_thal_likelihoods['Likelihood'].values
y_hat = df_thal_likelihoods['Label'].values

counts_thal,y_thal,sample_id,features,_ = load_thal_cross_lot_covs()
counts_thal = counts_thal[:-1]
y_thal = y_thal[:-1]
sample_id = sample_id[:-1]

index_homozygous_del = [4,5,11,18,19,24,32,36,57]
counts_thal_new = np.delete(counts_thal, index_homozygous_del, axis=0)
likelihoods_new = np.delete(likelihoods, index_homozygous_del)
y_new = np.delete(y_hat, index_homozygous_del)
snames_new = np.delete(snames,index_homozygous_del)  

                      Sample Name  Likelihood  Label
0     RDvGMpTHALD2d200302iM1-PB05   39.361303      0
1     RDvGMpTHALD2d200302iM1-PB06   39.298358      0
2     RDvGMpTHALD2d200302iM1-PB07   66.229424      0
3      RDvGMpTHALD2d200302iM1-TV2   38.732541      0
4  RDvGMpTHALD2d200302iM1-aTHAL31 -214.132134      0


In [9]:
def get_results(dist):
    def null_factory():
        return HeavyMixture(
                n_components=1,
                component_distribution=dist,
                n_init=3,
                max_iter=1000,
            )   

    def alt_factory():
        return HeavyMixture(
                n_components=2,
                component_distribution=dist,
                n_init=3,
                max_iter=1000,
            )   
    res = parametric_bootstrap_lrt(
            likelihoods_new,  
            null_model_factory=null_factory,
            alt_model_factory=alt_factory,
            n_bootstrap=10000,
            random_state=2021,
        )
    return res

In [4]:
res_gaussian = get_results('gaussian')
print('1')
res_laplace = get_results('laplace')
print('2')
res_student_t = get_results('student_t')
print('3')
res_hypsecant = get_results('hypsecant')
print('4')
res_gennorm = get_results('gennorm')


In [5]:
print("Gaussian p-value: %0.3f"%res_gaussian['p_value'])
print("Laplace p-value: %0.3f"%res_laplace['p_value'])
print("Student-T p-value: %0.3f"%res_student_t['p_value'])
print("Hyp p-value: %0.3f"%res_hypsecant['p_value'])
print("Gennorm p-value: %0.3f"%res_gennorm['p_value'])

Gaussian p-value: 0.000
Laplace p-value: 0.000
Student-T p-value: 0.005
Hyp p-value: 0.161
Gennorm p-value: 0.001


In [ ]:
a = 1

In [10]:
def get_results_subset(dist):
    def null_factory():
        return HeavyMixture(
                n_components=1,
                component_distribution=dist,
                n_init=3,
                max_iter=1000,
            )   

    def alt_factory():
        return HeavyMixture(
                n_components=2,
                component_distribution=dist,
                n_init=3,
                max_iter=1000,
            )   
    res1 = parametric_bootstrap_lrt(
            likelihoods_new[y_new==1],  
            null_model_factory=null_factory,
            alt_model_factory=alt_factory,
            n_bootstrap=10000,
            random_state=2021,
        )
    print('A')
    res2 = parametric_bootstrap_lrt(
            likelihoods_new[y_new==0],  
            null_model_factory=null_factory,
            alt_model_factory=alt_factory,
            n_bootstrap=10000,
            random_state=2021,
        )
    return res1,res2

In [6]:
res_gaussian_subgroup1,res_gaussian_subgroup2 = get_results_subset('gaussian')
print('Group 1')
res_laplace_subgroup1,res_laplace_subgroup2 = get_results_subset('laplace')
print('Group 2')
res_student_t_subgroup1,res_student_t_subgroup2 = get_results_subset('student_t')
print('Group 3')
res_hypsecant_subgroup1,res_hypsecant_subgroup2 = get_results_subset('hypsecant')
print('Group 4')
res_gennorm_subgroup1,res_gennorm_subgroup2 = get_results_subset('gennorm')
print('Group 5')


A
Group 1
A
Group 2
A
Group 3
A
Group 4
A


KeyboardInterrupt: 

In [7]:
results = {}
import pickle


results['res_gaussian_subgroup1'] = res_gaussian_subgroup1
results['res_laplace_subgroup1'] = res_laplace_subgroup1
results['res_student_t_subgroup1'] = res_student_t_subgroup1
results['res_hypsecant_subgroup1'] = res_hypsecant_subgroup1

results['res_gaussian_subgroup2'] = res_gaussian_subgroup2
results['res_laplace_subgroup2'] = res_laplace_subgroup2
results['res_student_t_subgroup2'] = res_student_t_subgroup2
results['res_hypsecant_subgroup2'] = res_hypsecant_subgroup2

with open('THAL_Res_P.p','wb') as f:
    pickle.dump(results,f)

In [4]:
import pickle

In [5]:
with open('THAL_Res_P.p','rb') as f:
    results = pickle.load(f)

In [ ]:
print("Gaussian p-value: %0.3f"%res_gaussian['p_value'])
print("Laplace p-value: %0.3f"%res_laplace['p_value'])
print("Student-T p-value: %0.3f"%res_student_t['p_value'])
print("Hyp p-value: %0.3f"%res_hypsecant['p_value'])
print("Gennorm p-value: %0.3f"%res_gennorm['p_value'])

In [7]:
print('Group 1 Gaussian p-value: ',results['res_gaussian_subgroup1']['p_value'])
print('Group 1 Laplace p-value: ',results['res_laplace_subgroup1']['p_value'])
print('Group 1 Student-T p-value: ',results['res_student_t_subgroup1']['p_value'])
print('Group 1 Hyp p-value: ',results['res_hypsecant_subgroup1']['p_value'])
#print(results['res_gennorm_subgroup1']['p_value'])

print('Group 2 Gaussian p-value: ',results['res_gaussian_subgroup2']['p_value'])
print('Group 2 Laplace p-value: ',results['res_laplace_subgroup2']['p_value'])
print('Group 2 Student-T p-value: ',results['res_student_t_subgroup2']['p_value'])
print('Group 2 Hyp p-value: ',results['res_hypsecant_subgroup2']['p_value'])
#print(results['res_gennorm_subgroup2'])

Group 1 Gaussian p-value:  0.0226977302269773
Group 1 Laplace p-value:  0.11788821117888211
Group 1 Student-T p-value:  0.058994100589941006
Group 1 Hyp p-value:  0.06929307069293071
Group 2 Gaussian p-value:  0.032596740325967405
Group 2 Laplace p-value:  0.65003499650035
Group 2 Student-T p-value:  0.18118188181181882
Group 2 Hyp p-value:  0.15458454154584542


In [11]:
res_gennorm_subgroup1,res_gennorm_subgroup2 = get_results_subset('gennorm')


A


In [12]:
print('Group 2 Gennorm p-value: ',res_gennorm_subgroup1['p_value'])
print('Group 2 Gennorm p-value: ',res_gennorm_subgroup2['p_value'])

Group 2 Gennorm p-value:  0.038696130386961305
Group 2 Gennorm p-value:  0.10078992100789921
